# ISCI — When a large cellular change is not control

**Researcher Track · Built with Claude: Life Sciences**

This executable walkthrough is for biologists, clinicians, and computational researchers. It explains the finding, reproduces its locked evidence from committed artifacts, and shows how a new Perturb-seq dataset enters the ISCI framework.

**Claim boundary:** ISCI does not declare therapeutic targets or clinical biomarkers. It tests whether a perturbation produces a directional, reproducible state shift beyond raw effect magnitude.

By the end, you will be able to distinguish perturbation size from state controllership, inspect all four evidence verdicts, and validate a new dataset contract without loading a large matrix.


In [1]:
from pathlib import Path
import json
import sys

import pandas as pd
import yaml
from IPython.display import display

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
assert (ROOT / "pyproject.toml").exists(), "Run from the repository or notebooks directory."
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from isci import load_dataset_spec, load_tabular_dataset, run_dataset, validate_dataset_spec

pd.set_option("display.max_colwidth", 90)


## 1. The medical question

A perturbation can make a cell change dramatically and still fail to control the clinically relevant state. The biological question is therefore not only **how much did the cell move?** but also where it moved, whether the direction repeated across donors and guides, and whether that information adds evidence after accounting for effect magnitude.

![Central controllership result](../figures/final/fig_central.png)


In [2]:
claims = json.loads((ROOT / "outputs/hackathon/claim_manifest.json").read_text())
primary = claims["claims"][0]
full = primary["metrics"]["authoritative_full_sample"]
oof = primary["metrics"]["leakage_free_oof"]

primary_metrics = pd.DataFrame([
    {
        "Estimand": "Full-sample pre-specified M → M+C",
        "ΔAUPRC": full["discovery_gain"],
        "95% CI": f'[{full["discovery_ci95"][0]:.3f}, {full["discovery_ci95"][1]:.3f}]',
        "Permutation p": None,
    },
    {
        "Estimand": "Leakage-free grouped OOF",
        "ΔAUPRC": oof["gain"],
        "95% CI": f'[{oof["ci95"][0]:.3f}, {oof["ci95"][1]:.3f}]',
        "Permutation p": oof["permutation_p"],
    },
])
display(primary_metrics)


,Estimand,ΔAUPRC,95% CI,Permutation p
0,Full-sample pre-specified M → M+C,0.3570,"[0.117, 0.538]",NaN
1,Leakage-free grouped OOF,0.2151,"[0.074, 0.560]",0.01


### Clinical-language interpretation

Among perturbations with comparable overall effect size, canonical T-cell state regulators were more likely to produce a **focused functional shift** and a **repeatable direction across donors**. The conservative out-of-fold estimate remained positive.

The two estimates answer related but different questions. They must not be averaged or presented as interchangeable.


## 2. A trustworthy result includes where it fails

The project uses four explicit verdicts. A failed generalization test does not erase a bounded primary result; it defines its scope. Missing inputs are not converted into a negative biological result.


In [3]:
verdict_rows = []
for claim in claims["claims"]:
    verdict_rows.append({
        "Question": claim["short_label"],
        "Verdict": claim["verdict"],
        "Scope": claim["scope"],
        "Do not claim": claim["prohibited_overclaim"],
    })
display(pd.DataFrame(verdict_rows))


,Question,Verdict,Scope,Do not claim
0,Canonical T-cell state controllers,PASS,Detectable-effect perturbations and canonical axis-defining T-cell regulators.,Broad functional-regulator discovery or therapeutic target nomination.
1,Broad external functional regulators,FAIL,Twenty external non-marker regulators with matched negatives.,The failed broad class invalidates the bounded canonical-regulator result.
2,CAR-T clinical response,NULL,Eighty-seven labeled patients across nine public studies.,Proof of no biological effect or a validated clinical biomarker.
3,scGPT zero-shot corroboration,NOT-EVALUABLE,Current machine state and the pre-specified expression-profile input contract.,"A scientific null, failed foundation model, or negative biological result."


### Stress test and scope map

![Positive-set stress test](../figures/positive_set_stress_test.png)

![Cross-system scope](../figures/cci_scope_4systems.png)


In [4]:
oof_evidence = json.loads((ROOT / "outputs/isci_oof_incremental.json").read_text())
stress = json.loads((ROOT / "outputs/positive_set_stress_test.json").read_text())
external = stress["external_screen_test"]

robustness = pd.DataFrame([
    {
        "Test": "Leakage-free OOF",
        "Result": "PASS",
        "ΔAUPRC": oof_evidence["oof_gain"],
        "95% CI": oof_evidence["hierarchical_bootstrap"]["ci95"],
        "p": oof_evidence["within_block_permutation"]["perm_p"],
    },
    {
        "Test": "Independent functional-regulator set",
        "Result": external["verdict"],
        "ΔAUPRC": external["gain"],
        "95% CI": external["ci"],
        "p": external["p_gain"],
    },
])
display(robustness)


,Test,Result,ΔAUPRC,95% CI,p
0,Leakage-free OOF,PASS,0.215100,"[0.0737, 0.5604]",0.010
1,Independent functional-regulator set,FAIL,-0.280728,"[-0.47638489435959946, -0.07342890735600327]",0.005


## 3. How Claude Science changed the research

Claude was not used to manufacture a positive result. It helped maintain a correction loop: formulate, challenge, falsify, reformulate, and bind each claim to evidence.


In [5]:
correction_history = pd.DataFrame([
    {"Stage": "Initial formulation", "Question": "Does a multi-component index beat magnitude?", "Outcome": "Failed: known regulators had much larger effects."},
    {"Stage": "Scientific correction", "Question": "Does direction/reproducibility add signal conditional on magnitude?", "Outcome": "Supported for canonical axis-defining regulators."},
    {"Stage": "Leakage audit", "Question": "Were matching and residualization repeated inside every fold?", "Outcome": "Pipeline corrected; OOF estimate decreased to a more honest +0.215."},
    {"Stage": "Generalization challenge", "Question": "Does the signal recover broad external functional regulators?", "Outcome": "No. The external test failed and bounded the claim."},
])
display(correction_history)


,Stage,Question,Outcome
0,Initial formulation,Does a multi-component index beat magnitude?,Failed: known regulators had much larger effects.
1,Scientific correction,Does direction/reproducibility add signal conditional on magnitude?,Supported for canonical axis-defining regulators.
2,Leakage audit,Were matching and residualization repeated inside every fold?,Pipeline corrected; OOF estimate decreased to a more honest +0.215.
3,Generalization challenge,Does the signal recover broad external functional regulators?,No. The external test failed and bounded the claim.


## 4. The reusable framework

The notebook is the teaching surface; the framework lives in typed Python interfaces and a versioned DatasetSpec.

**Dataset YAML → contract validation → physical inspection → canonical table or stream → feature runner → auditable outputs**

Supported execution boundary:

- **controller_features:** inspect and rank precomputed features;
- **long_effects:** extract features and rank from CSV or Parquet;
- **anndata_effects:** extract features and rank from backed H5AD with bounded-memory blocks.


In [6]:
example_path = ROOT / "examples/dataset_spec/mini_long_effects.yaml"
raw_spec = yaml.safe_load(example_path.read_text())
contract = validate_dataset_spec(raw_spec, repo_root=ROOT, check_paths=True)
spec = load_dataset_spec(example_path, repo_root=ROOT, check_paths=True)
physical = load_tabular_dataset(spec, repo_root=ROOT)
analysis = run_dataset(spec, repo_root=ROOT)

framework_check = pd.DataFrame([
    {
        "Layer": "DatasetSpec declaration",
        "Valid": contract.valid,
        "Capability": contract.capability.value,
        "Meaning": "Required fields are declared; data have not yet proven the claim.",
    },
    {
        "Layer": "Physical data inspection",
        "Valid": physical.inspection.evaluable,
        "Capability": physical.inspection.runtime_capability.value,
        "Meaning": "Observed coverage is too small for a confirmatory analysis.",
    },
    {
        "Layer": "Feature extraction + ranking",
        "Valid": analysis.completed,
        "Capability": analysis.status,
        "Meaning": "The tiny fixture lacks enough axis and replicate coverage; no ranking is invented.",
    },
])
display(framework_check)


,Layer,Valid,Capability,Meaning
0,DatasetSpec declaration,True,CONFIRMATORY_DECLARED,Required fields are declared; data have not yet proven the claim.
1,Physical data inspection,True,DIAGNOSTIC_ONLY,Observed coverage is too small for a confirmatory analysis.
2,Feature extraction + ranking,False,FEATURE_EXTRACTION_NOT_EVALUABLE,The tiny fixture lacks enough axis and replicate coverage; no ranking is invented.


In [7]:
commands = pd.DataFrame([
    {"Goal": "Validate contract and paths", "Command": "isci validate dataset.yaml"},
    {"Goal": "Validate before data download", "Command": "isci validate dataset.yaml --structure-only"},
    {"Goal": "Inspect CSV, Parquet, or H5AD", "Command": "isci inspect dataset.yaml"},
    {"Goal": "Scan H5AD layers by blocks", "Command": "isci inspect dataset.yaml --scan-values --block-rows 64"},
    {"Goal": "Extract and rank any supported layout", "Command": "isci run dataset.yaml --output-dir outputs/my_dataset"},
    {"Goal": "Limit H5AD matrix reads", "Command": "isci run dataset.yaml --block-rows 32"},
])
display(commands)


,Goal,Command
0,Validate contract and paths,isci validate dataset.yaml
1,Validate before data download,isci validate dataset.yaml --structure-only
2,"Inspect CSV, Parquet, or H5AD",isci inspect dataset.yaml
3,Scan H5AD layers by blocks,isci inspect dataset.yaml --scan-values --block-rows 64
4,Extract and rank any supported layout,isci run dataset.yaml --output-dir outputs/my_dataset
5,Limit H5AD matrix reads,isci run dataset.yaml --block-rows 32


### Exercise: bring a dataset

Copy the example DatasetSpec, change only paths and mappings, then predict the runtime capability before inspection. A useful answer is not always confirmatory. DIAGNOSTIC_ONLY or NOT_EVALUABLE can be the scientifically correct outcome.


In [8]:
BYOD_SPEC = None  # Example: ROOT / "config/my_dataset.yaml"

if BYOD_SPEC is None:
    print("Set BYOD_SPEC to validate your own public Perturb-seq dataset. No external data were opened.")
else:
    candidate = load_dataset_spec(BYOD_SPEC, repo_root=ROOT, check_paths=True)
    print(f"Validated: {candidate.dataset.label} ({candidate.input.layout})")
    print("Next: inspect every downgrade, then run isci run with a local output directory.")


Set BYOD_SPEC to validate your own public Perturb-seq dataset. No external data were opened.


## 5. Auditability is part of the result

Every public claim is bound to a source artifact, data hash, axes hash, Git revision, command, and release gate. The framework never commits raw H5AD or H5MU inputs.


In [9]:
readiness = json.loads((ROOT / "outputs/hackathon/readiness_report.json").read_text())
readiness_summary = pd.DataFrame([
    {
        "Package status": readiness["status"],
        "Automated gates passed": f'{sum(readiness["checks"].values())}/{len(readiness["checks"])}',
        "Raw or secret tracked files": len(readiness["details"]["forbidden_tracked_files"]),
        "Public local-path violations": len(readiness["details"]["local_path_violations"]),
    }
])
display(readiness_summary)


,Package status,Automated gates passed,Raw or secret tracked files,Public local-path violations
0,AUTOMATED_GATES_PASS_HUMAN_GATES_PENDING,19/19,0,0


## 6. What is proven — and what is not

**Supported:** a magnitude-conditional signal for canonical axis-defining regulators in the primary CD4+ T-cell screen, with a positive leakage-free OOF estimate.

**Not supported:** broad functional-regulator generalization, clinical-response prediction, therapeutic desirability, or a universal immune-cell invariant.

**Framework boundary today:** controller-feature tables, long-effect CSV/Parquet, and backed H5AD run end-to-end through the same frozen conditional method. Missing axis coverage or replication remains NOT_EVALUABLE rather than being imputed. A new dataset still requires independent scientific adjudication before any biological PASS.


## 7. Prospective experiment

The framework converts uncertainty into an experiment: a proposed 25-gene, 54-guide panel across 8–12 donors, with guide sequence and QC as a stop gate before synthesis.

![Controller convergence and prospective space](../figures/controller_convergence_quadrant.png)

The next validation increment is an independent public H5AD smoke run not used during adapter development. That tests portability of the framework; it does not change the frozen scientific claim.


## Take-home message

**Magnitude tells us how far a cell moved. Controllership asks whether it moved in the intended functional direction, whether that direction repeated, and whether the evidence survives attempts to falsify it.**

ISCI is both a bounded biological finding and an auditable framework for testing that question in new Perturb-seq datasets.
